# SmartBasket · Notebook 05 — Are our recommendations better than best-sellers?

**Team Frenchies** · Introduction to Big Data · Week 11

**The test.** The rules were mined on the *prior* orders. Instacart also gives **131,209 other orders** (the *train* file) that A-Priori never saw. For each of these orders we **hide one product at random**, give the rest of the basket to each method, and check whether the hidden product is in its top 5 or top 10 recommendations (**hit-rate@k**).

| Method | How it recommends |
|---|---|
| Best-sellers (baseline) | the most bought products that are not already in the basket |
| Rules by lift | our A-Priori rules, ranked by lift (what the agent uses) |
| Rules by confidence | the same rules, ranked by confidence |
| Rules + best-sellers | rules by lift first, then best-sellers to fill the list |

> 🇫🇷 *C'est la comparaison à une baseline que le prof demande dans chaque lab. Si nos règles font moins bien que « les produits les plus vendus », c'est aussi un résultat : il faut le dire et l'expliquer.*

In [ ]:
# 🇫🇷 Imports et paramètres
import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
KS = [5, 10]
OUT = "out/eval_week11"
os.makedirs(OUT, exist_ok=True)

BASE = "https://raw.githubusercontent.com/arthurcomet/smartbasket-frenchies/main/"
RULE_FILES = {
    "A-Priori 0.05% (notebook 02)": os.environ.get("SB_RULES_005", BASE + "out/apriori_20261118/rules_full_support_0p05.csv"),
    "PCY 0.01% (notebook 03)": os.environ.get("SB_RULES_001", BASE + "out/pcy_week11/rules_full_support_0p01.csv"),
}

In [ ]:
# 🇫🇷 Données : d'abord Google Drive (dossier « instacart »), sinon Kaggle
DRIVE_DIR = "/content/drive/MyDrive/instacart"
DATA_DIR = os.environ.get("SB_DATA_DIR")          # local test only

if DATA_DIR is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        if os.path.exists(os.path.join(DRIVE_DIR, "order_products__prior.csv")):
            DATA_DIR = DRIVE_DIR
            print("Data found on Google Drive")
        else:
            print("No 'instacart' folder with the CSV files on Drive, trying Kaggle")
    except Exception as e:
        print("Google Drive not available:", type(e).__name__)

if DATA_DIR is None:
    try:
        from google.colab import userdata
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        pass
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("psparks/instacart-market-basket-analysis")

print("DATA_DIR =", DATA_DIR)

In [ ]:
# 🇫🇷 Best-sellers (calculés sur prior) et paniers de test (train, jamais vus par A-Priori)
t0 = time.time()
prior_products = pd.read_csv(os.path.join(DATA_DIR, "order_products__prior.csv"),
                             usecols=["product_id"], dtype={"product_id": "int32"})["product_id"]
BEST_SELLERS = prior_products.value_counts().index.to_numpy()[:200]
del prior_products

train = pd.read_csv(os.path.join(DATA_DIR, "order_products__train.csv"),
                    usecols=["order_id", "product_id"], dtype={"order_id": "int32", "product_id": "int32"})
baskets = [np.unique(b) for b in train.groupby("order_id")["product_id"].apply(np.array)]
baskets = [b for b in baskets if len(b) >= 2]
rng = np.random.default_rng(SEED)
hidden = np.array([rng.choice(b) for b in baskets])
print(f"test baskets with 2+ products: {len(baskets):,}   loaded in {time.time() - t0:.1f} s")

In [ ]:
# 🇫🇷 Les 4 méthodes et le calcul du hit-rate
def load_rules(path):
    try:
        r = pd.read_csv(path)
    except Exception as e:
        print(f"  not available ({type(e).__name__}): {path}")
        return None
    r = r[r["lift"] > 1]
    return {a: (g["b_id"].to_numpy(), g["lift"].to_numpy(), g["confidence"].to_numpy())
            for a, g in r.groupby("a_id")}, set(r["b_id"].unique())

def best_sellers(rest, k):
    out = []
    for p in BEST_SELLERS:
        if p not in rest:
            out.append(p)
            if len(out) == k:
                break
    return out

def by_rules(rest, index, k):
    """Top-k by lift and top-k by confidence, in one pass over the basket's rules."""
    lift, conf = {}, {}
    for a in rest:
        if a in index:
            bs, lifts, confs = index[a]
            for b, l, c in zip(bs.tolist(), lifts.tolist(), confs.tolist()):
                if b in rest:
                    continue
                if l > lift.get(b, 0):
                    lift[b] = l
                if c > conf.get(b, 0):
                    conf[b] = c
    top = lambda d: [b for b, _ in sorted(d.items(), key=lambda x: -x[1])[:k]]
    return top(lift), top(conf)

def evaluate(name, index, reachable):
    K = max(KS)
    hits = {m: {k: 0 for k in KS} for m in ["Best-sellers", "Rules by lift", "Rules by confidence", "Rules + best-sellers"]}
    covered = covered_hits = 0
    for b, h in zip(baskets, hidden):
        rest = set(b.tolist()) - {int(h)}
        h = int(h)
        recs = {"Best-sellers": best_sellers(rest, K)}
        if index is not None:
            lift, conf = by_rules(rest, index, K)
            recs["Rules by lift"] = lift
            recs["Rules by confidence"] = conf
            fill = [p for p in recs["Best-sellers"] if p not in lift]
            recs["Rules + best-sellers"] = (lift + fill)[:K]
            if lift:
                covered += 1
                covered_hits += h in lift[:10]
        for m, r in recs.items():
            for k in KS:
                hits[m][k] += h in r[:k]
    n = len(baskets)
    rows = [{"rules": name, "method": m, **{f"hit@{k}_%": round(100 * hits[m][k] / n, 2) for k in KS}}
            for m in hits if index is not None or m == "Best-sellers"]
    extra = {"rules": name, "test_baskets": n,
             "baskets_with_a_rule_recommendation_%": round(100 * covered / n, 2) if index else None,
             "hit@10_when_rules_recommend_%": round(100 * covered_hits / covered, 2) if covered else None,
             "hidden_product_is_recommendable_%": round(100 * float(np.isin(hidden, list(reachable)).mean()), 2) if index else None}
    return rows, extra

In [ ]:
# 🇫🇷 Évaluation de chaque fichier de règles (2 à 5 minutes)
results, details = [], []
for name, path in RULE_FILES.items():
    print(name)
    loaded = load_rules(path)
    if loaded is None:
        continue
    t0 = time.time()
    rows, extra = evaluate(name, *loaded)
    extra["seconds"] = round(time.time() - t0, 1)
    results += rows
    details.append(extra)
    print("  ", extra)

table = pd.DataFrame(results)
table.to_csv(f"{OUT}/hit_rates.csv", index=False)
table

In [ ]:
# 🇫🇷 Graphique : hit-rate@10 de chaque méthode
fig, ax = plt.subplots(figsize=(10, 4.5))
labels = table["rules"].str.split(" \\(").str[0] + "\n" + table["method"]
colors = ["#888888" if m == "Best-sellers" else "#1F77B4" for m in table["method"]]
ax.barh(labels, table["hit@10_%"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("hit-rate@10 (% of test baskets where the hidden product is in the top 10)")
ax.set_title("Recommendations vs best-sellers on 131k unseen orders")
for i, v in enumerate(table["hit@10_%"]):
    ax.text(v, i, f" {v:.1f}%", va="center")
plt.tight_layout(); plt.savefig(f"{OUT}/hit_rates.png", dpi=120); plt.show()

In [ ]:
# 🇫🇷 Preuves pour le rapport de la semaine 11
evidence = {"seed": SEED, "test_file": "order_products__train.csv", "ks": KS,
            "hit_rates": results, "coverage": details}
with open(f"{OUT}/evidence_week11_eval.json", "w") as f:
    json.dump(evidence, f, indent=2)
pd.DataFrame(details)

## What to say at the presentation

Fill in the brackets with the numbers printed above.

> *"To test our recommendations, we used 131,000 orders that A-Priori never saw. In each one we hid one product and checked if it came back in the top 10. Best-sellers find it [X] % of the time, our rules ranked by lift [Y] %, and rules plus best-sellers [Z] %. Our rules only recommend something for [C] % of the baskets: that is why lowering the support with PCY matters."*

> 🇫🇷 *À regarder en priorité : (1) est-ce que « Rules + best-sellers » bat « Best-sellers » seul ? (2) est-ce que le fichier PCY 0,01 % fait mieux que 0,05 %, grâce à plus de produits couverts ? (3) lift contre confiance : la confiance recommande plus souvent des produits populaires, donc elle peut gagner au hit-rate tout en étant moins intéressante. Si c'est le cas, c'est un vrai point de discussion pour le rapport.*

In [ ]:
# 🇫🇷 Télécharger tous les résultats en un zip (à mettre ensuite sur GitHub, dans le dossier out/)
import shutil
zip_path = shutil.make_archive(OUT.rstrip("/").split("/")[-1], "zip", root_dir="out", base_dir=OUT.rstrip("/").split("/")[-1])
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("created", zip_path)